## CLUSTERING ANALYSIS 

```
Understanding and Implementing K-Means and DBSCAN Algorithms

Objective:
The objective of this assignment is to introduce to various clustering algorithms, including K-Means, hierarchical, and DBSCAN, and provide hands-on experience in applying these techniques to a real-world dataset.
Datasets :

Data Preprocessing:
1.	Preprocess the dataset to handle missing values, remove outliers, and scale the features if necessary.
2.	Perform exploratory data analysis (EDA) to gain insights into the distribution of data and identify potential clusters.
3.	Use multiple visualizations to understand the hidden patterns in the dataset

Implementing Clustering Algorithms:
•	Implement the K-Means and DBSCAN algorithms using a programming language such as Python with libraries like scikit-learn.
•	Apply each clustering algorithm to the pre-processed dataset to identify clusters within the data.
•	Experiment with different parameter settings for K-means (Elbow curve for different K values) and DBSCAN (e.g., epsilon, minPts) and evaluate the clustering results.

Cluster Analysis and Interpretation:
•	Analyse the clusters generated by each clustering algorithm and interpret the characteristics of each cluster. Write you insights in few comments.


Visualization:
Visualize the clustering results using scatter plots or other suitable visualization techniques.
Plot the clusters with different colours to visualize the separation of data points belonging to different clusters.
Evaluation and Performance Metrics:
Evaluate the quality of clustering using internal evaluation metrics such as silhouette score for K-Means and DBSCAN.

```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# loading the file into our enviroment
dataset = pd.read_excel("EastWestAirlines.xlsx",sheet_name ='data')

#Working on a cloned copy
df = dataset.copy()

# Understanding data set
print("\n<----------INFO----------->\n")
print(df.info())

print("\n<-----------DESCRIBE ONLY NUMERICAL---------->\n")
print(df.describe())

print("\n<---------DESCRIBE ALL NUMERICAL AND CATEGORICAL--------->\n")
print(df.describe(include='all'))

print("\n<---------MISSING VALUES--------->\n")
print(df.isnull().sum())

In [ ]:
#Id is identifier so removing 
df = df.drop(columns=["ID#"])

# Defining categorical and numerical columns
categorical_cols = ["cc1_miles", "cc2_miles", "cc3_miles", "Award?"]
numerical_cols = [col for col in df.columns if col not in categorical_cols]

# Print results
print("Categorical Columns:", categorical_cols)
print("Numerical Columns:", numerical_cols)
 
# Split datasets
df_categorical = df[categorical_cols]
df_numerical = df[numerical_cols]
print(df_categorical)
print(df_numerical)

## Exploratory Data Analysis (EDA):

In [ ]:
# Analysing the numerical columns for 
for col in numerical_cols:
    plt.figure(figsize=(18,9))

    plt.subplot(1,2,1)
    sns.histplot(df[col],bins=20,kde='True')
    plt.title(f"Histogram for {col}")

    plt.subplot(1,2,2)
    sns.boxplot(df[col])
    plt.title(f"Boxplot for {col}")

    plt.show()

In [ ]:
# Columns considered for the outliers are here
considered_cols = ['Balance','Bonus_miles','Flight_miles_12mo','Days_since_enroll']

outliers_summary = {}
for col in considered_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_limit = 1 - IQR*1.5
    upper_limit = 1 + IQR*1.5
    outliers = (df[col] < lower_limit) | (df[col] > upper_limit)
    outliers_summary[col] = outliers.sum()
    df[col] = np.clip(df[col], lower_limit, upper_limit) # Clipping the outliers 

outlier_count = pd.DataFrame.from_dict(outliers_summary,orient="index", columns=["Outlier_count"])

print("\n<---------No of outliers in each columns---------->\n")
print(outlier_count)

In [ ]:
for col in considered_cols:
    plt.figure(figsize=(18,9))

    plt.subplot(1,2,1)
    sns.histplot(df[col],bins=20,kde='True')
    plt.title(f"Histogram for {col}")

    plt.subplot(1,2,2)
    sns.boxplot(df[col])
    plt.title(f"Boxplot for {col}")

    plt.show()

In [ ]:
# There is no null values in any feature so we can perform the scaling directly on the numerical and categorical columns
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer



# preprocessing scaled numerical and encoded categorical variables
preprocessor = ColumnTransformer(
    transformers = [
        ("num",StandardScaler(),numerical_cols),
         ("cate",OneHotEncoder(drop='first'),categorical_cols)   
 ]
)

# Applying preprocessing 
X_preprocessed= preprocessor.fit_transform(df)

# getting encoded categorical cols 
encoded_cate_cols = preprocessor.named_transformers_['cate'].get_feature_names_out(categorical_cols)
all_cols = numerical_cols + list(encoded_cate_cols)
df_processed = pd.DataFrame(X_preprocessed.toarray() if hasattr(X_preprocessed, "toarray") else X_preprocessed, columns = all_cols)

print("\n<----------Processed Dta is given by---------->\n")
print(df_processed.head())


In [ ]:
# Distribution of numerical columns
df_processed.hist(figsize=(15,10), bins=30, edgecolor = 'black')
plt.suptitle("Feature Distributions for Processed Data", fontsize=16)
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(15,10))
sns.heatmap(df_processed.corr(),cmap='coolwarm', annot=True)
plt.title("Correlation Heatmap", fontsize=16)
plt.show()

# There us no much correlationship between the numerical values so we can keep all the features 

In [ ]:
# PCA for visualization 
from sklearn.decomposition import PCA

pca = PCA(n_components=2) 
pca_result = pca.fit_transform(df_processed)

pca_df = pd.DataFrame(data=pca_result, columns=["PC1","PC2"])
plt.figure(figsize=(8,6))
sns.scatterplot(x="PC1",y="PC2",data= pca_df, alpha=0.7)
plt.title("PCA projection (Unclustered Data)",fontsize=16)
plt.show()

## Implementing Clustering Algorithms:

In [ ]:
# K-Means implementation
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

wcss = [] # within cluster sum of square
sil_scores = []

for k in range(2,11):
    kmeans = KMeans(n_clusters=k, random_state = 42, n_init=10)
    kmeans.fit_transform(df_processed)
    wcss.append(kmeans.inertia_)
    sil_scores.append(silhouette_score(df_processed,kmeans.labels_))

# Elbow score 
plt.figure(figsize=(15,10))
plt.plot(range(2,11),wcss, marker='o')
plt.xlabel("Number of Clusters (k)")
plt.ylabel("WCSS")
plt.title("Elbow method (WCSS)")
plt.show()


# Silhoutte score
plt.figure(figsize=(15,10))
plt.plot(range(2,11),sil_scores,marker='o')
plt.title("Sillhoute score for k-means ")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Sillhoutte Score")
plt.show()

In [ ]:
print("Wcss values",wcss)
print("silhouette scores", sil_scores)

## Analysis from graph:
```
i. Here, elbow method suggest there are about 6-7 meaningful sub groups.
ii. The silhoutte peak at k = 2 indicates that, in terms of cluster separation and cohesion, forcing exactly 2 clusters yields clean separation. However, the improvement beyond 2 2 clusters diminishes, and around 6-9 the silhouette scores are not dramatically worse than 2, but they are not better either.
    hence, i am choosing 2 clusters .

```

In [ ]:
best_k = 2
kmeans = KMeans(n_clusters=2,random_state=42,n_init=10)
df_processed["KMeans_Cluster"] = kmeans.fit_predict(df_processed)
print("Cluster counts:\n", df_processed["KMeans_Cluster"].value_counts())

In [ ]:
# DBSCAN Implementation
from sklearn.cluster import DBSCAN

# trying different episilon values and min_smaple vlaues
eps_values = [0.5,1.0,1.5,2.0]
min_pts = [3,5,10]
for eps in eps_values:
    for pts in min_pts:
        db = DBSCAN(eps=eps, min_samples=pts)
        labels = db.fit_predict(df_processed)

        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)

        if n_clusters > 1:
            sil = silhouette_score(df_processed,labels)
        else:
            sil = -1
        print(f"DBSCAN -> eps={eps}, min_samples={pts}, clusters={n_clusters}, silhouette={sil:.3f}")

    

In [ ]:
# Final DBSCAN with choosen charachters 
db_final = DBSCAN(eps=1.0,min_samples=10)
df_processed["DBSCAN_Cluster"] = db_final.fit_predict(df_processed)
print("Cluster counts:\n", df_processed["DBSCAN_Cluster"].value_counts())

## Cluster Analysis and Interpretation:
```
KMeans:
Reveals two broad well-separated groups with sizes 2348 and 1651. It's good for a hogh-level dichotomy.

DBSCAN (eps=1.0, min_samples = 10):
Yields multiple core clusters and a sizable amount of noise (769 points). Main cluster 0 is larger(1842) with several smaller clusters. It is suitable for density-based substructure but watch noise and interpretability.

Practical Applications:
i. choose the kmeans when you need clear group segmentations.Like, here k=2.
ii. If you want handle noise then explose DBSCAN which is better in noise removal.

```

## Visualization:

In [ ]:
#using PCA for visualization purpose
plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.scatter(pca_result[:,0],pca_result[:,1], c=kmeans.labels_,cmap="tab10",s=20)
plt.xlabel("PCA1")
plt.ylabel("PCA2")
plt.title("Kmeans Clusters")

plt.subplot(1,2,2)
plt.scatter(pca_result[:,0],pca_result[:,1],c=db_final.labels_, cmap="tab20",s=20)
plt.title("DBSCAN Clusters")
plt.xlabel("PCA1")
plt.ylabel("PCA2")

plt.show()

## Evaluation and performance Metrices

In [ ]:
# sillhouette_score for the kmean clustering 

sil_kmeans = silhouette_score(df_processed,kmeans.labels_)
print("Silhouette score for kmean clustering:", sil_kmeans)

#Sillhouette_score for the DBSCAN clustering
sil_dbscan = silhouette_score(df_processed,db_final.labels_)
print("Silhouette score for DBSCAN clustering:", sil_dbscan)


                        <-------------------------------COMPLETE----------------------------->